# CREP Tensor & Brain Criticality

This notebook demonstrates how the four CREP diagnostic components (C, R, E, P)
respond to the branching ratio σ_b as it sweeps from subcritical → critical → supercritical.

The key result: **Γ = arctanh(σ_b / K) / σ** peaks near 0.251 at σ_b = 1.0 (criticality).

Components:
- **C**: AR(1) autocorrelation (critical slowing down)
- **R**: Power-law exponent proximity to τ = 3/2
- **E**: Pairwise correlation excess (emergence)
- **P**: Permutation entropy proximity (intermediate disorder at criticality)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from neural_avalanche_utac.spike_train import SpikeTrainConfig, SpikeTrainGenerator
from neural_avalanche_utac.crep_neural import NeuralCREPTensor

In [ ]:
# Sweep branching ratio from 0.3 to 1.8
sigma_values = np.linspace(0.3, 1.8, 16)
crep = NeuralCREPTensor()

C_vals, R_vals, E_vals, P_vals, Gamma_vals = [], [], [], [], []

for i, sigma_b in enumerate(sigma_values):
    cfg = SpikeTrainConfig(n_neurons=200, duration_s=120.0, dt_ms=2.0,
                           branching_ratio=sigma_b, seed=42 + i)
    spikes = SpikeTrainGenerator(cfg).generate()['spikes']
    out = crep.compute(spikes)
    C_vals.append(out['C']); R_vals.append(out['R'])
    E_vals.append(out['E']); P_vals.append(out['P'])
    Gamma_vals.append(out['Gamma'])
    print(f'σ_b={sigma_b:.2f}  C={out["C"]:.3f}  R={out["R"]:.3f}  '
          f'E={out["E"]:.3f}  P={out["P"]:.3f}  Γ={out["Gamma"]:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

# Theoretical Gamma curve
sig_th = np.linspace(0.01, 1.99, 200)
gamma_th = np.arctanh(sig_th / 2.0) / 2.2

plots = [
    ('C — Autocorrelation', C_vals, 'steelblue'),
    ('R — Power-law Proximity', R_vals, 'darkorange'),
    ('E — Emergence', E_vals, 'forestgreen'),
    ('P — Permutation Entropy', P_vals, 'purple'),
    ('Γ — CREP Tensor', Gamma_vals, 'crimson'),
]
for ax, (title, vals, color) in zip(axes, plots):
    ax.plot(sigma_values, vals, 'o-', color=color, lw=2)
    ax.axvline(1.0, color='gray', ls='--', lw=1, label='σ_b=1 (critical)')
    if 'Γ' in title:
        ax.plot(sig_th, gamma_th, 'k--', lw=1.5, alpha=0.6, label='arctanh(σ_b/2)/2.2')
        ax.axhline(0.251, color='crimson', ls=':', lw=1.5, label='Γ*=0.251')
    ax.set_xlabel('Branching ratio σ_b')
    ax.set_title(title)
    ax.legend(fontsize=8)

axes[-1].axis('off')
plt.suptitle('CREP Components vs Branching Ratio (GenesisAeon Package 20)', fontsize=13)
plt.tight_layout()
plt.show()